# Eval-Only Results

直接加载现有基模或 `output/` 中已有 checkpoint 做评估和预测展示。这个 notebook 不运行 pretrain，也不运行 LoRA / head fine-tune。

In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from datetime import datetime
from pathlib import Path

MPLCONFIGDIR = Path('/tmp/matplotlib-ml-project')
MPLCONFIGDIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(MPLCONFIGDIR))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'src').is_dir() and (candidate / 'notebook').is_dir():
            return candidate
    raise RuntimeError('Cannot locate project root from current working directory')

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
if '' not in sys.path:
    sys.path.insert(0, '')

PYTHON = sys.executable
EVAL_ROOT = PROJECT_ROOT / 'output' / f'notebook_eval_{datetime.now():%Y%m%d_%H%M%S}'
TSF_PATH = PROJECT_ROOT / 'data' / 'extracted' / 'tourism_monthly_dataset.tsf'
SEED = 42
PREDICTION_LENGTH = 24
CONTEXT_LENGTH = 128
N_SERIES = 73

def rel(path: str | Path) -> str:
    path = Path(path)
    try:
        return str(path.relative_to(PROJECT_ROOT))
    except ValueError:
        return str(path)

def require_path(path: str | Path, marker: str | None = None) -> Path:
    path = Path(path)
    if not path.is_absolute():
        path = PROJECT_ROOT / path
    target = path / marker if marker else path
    if not target.exists():
        raise FileNotFoundError(f'Missing required file: {rel(target)}')
    return path

def run_process(args: list[str], run_dir: Path) -> None:
    run_dir.mkdir(parents=True, exist_ok=True)
    log_path = run_dir / 'eval.log'
    cmd = [PYTHON, *map(str, args)]
    print(' '.join(cmd))
    with log_path.open('w') as log:
        proc = subprocess.Popen(
            cmd,
            cwd=PROJECT_ROOT,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end='')
            log.write(line)
        code = proc.wait()
    if code != 0:
        raise RuntimeError(f'Command failed with exit code {code}: {rel(log_path)}')

def show_results(run_dir: Path) -> None:
    summaries = sorted(run_dir.rglob('results_summary.json'))
    if not summaries:
        print(f'No results_summary.json under {rel(run_dir)}')
        return
    for summary in summaries:
        print(f'\n{rel(summary)}')
        display(pd.DataFrame(json.loads(summary.read_text())))

def _to_float_array(x):
    return np.asarray(x, dtype=float).reshape(-1)

def _prediction_files(run_dir: Path) -> list[Path]:
    files: list[Path] = []
    for name in ('predictions.npz', 'zeroshot_predictions.npz'):
        files.extend(sorted(run_dir.rglob(name)))
    return files

def plot_predictions(run_dir: Path, title: str, max_panels: int = 4) -> None:
    files = _prediction_files(run_dir)
    if not files:
        print(f'No prediction npz under {rel(run_dir)}')
        return
    rng = np.random.default_rng(SEED)
    chosen = files[:max_panels]
    fig, axes = plt.subplots(len(chosen), 1, figsize=(10, 3.2 * len(chosen)), squeeze=False)
    axes = axes[:, 0]
    for ax, npz_path in zip(axes, chosen):
        z = np.load(npz_path, allow_pickle=True)
        n = len(z['contexts'])
        i = int(rng.integers(0, n))
        ctx = _to_float_array(z['contexts'][i])
        point = _to_float_array(z['point_forecasts'][i])
        actual = _to_float_array(z['actuals'][i]) if 'actuals' in z.files else None
        quants = np.asarray(z['quantile_forecasts'][i], dtype=float) if 'quantile_forecasts' in z.files else None
        ctx_tail = ctx[-min(len(ctx), 96):]
        x_ctx = np.arange(-len(ctx_tail), 0)
        x_pred = np.arange(len(point))
        ax.plot(x_ctx, ctx_tail, label='context', color='#334155')
        if actual is not None:
            ax.plot(x_pred, actual, label='actual', color='#111827', linewidth=2)
        ax.plot(x_pred, point, label='forecast', color='#dc2626', linewidth=2)
        if quants is not None and quants.ndim == 2 and quants.shape[1] >= 2:
            ax.fill_between(x_pred, quants[:, 0], quants[:, -1], color='#fca5a5', alpha=0.35, label='q10-q90')
        ax.axvline(-0.5, color='#94a3b8', linestyle='--', linewidth=1)
        ax.set_title(f'{title} | {rel(npz_path)} | sample={i}')
        ax.legend(loc='best')
        ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()

def make_chronos_eval_tsf(run_dir: Path) -> tuple[Path, str]:
    from src.chronos_finetune import parse_tsf, write_tsf

    series, freq = parse_tsf(str(TSF_PATH))
    n_eval = max(1, int(len(series) * 0.2))
    eval_tsf = run_dir / 'eval_data.tsf'
    write_tsf(series[-n_eval:], freq, str(eval_tsf))
    return eval_tsf, freq

def eval_chronos_zeroshot(run_dir: Path) -> None:
    import torch
    from chronos import ChronosPipeline
    from src.chronos_finetune import DEVICE, ZERO_SHOT_LABEL, evaluate_model

    run_dir.mkdir(parents=True, exist_ok=True)
    eval_tsf, freq = make_chronos_eval_tsf(run_dir)
    pipeline = ChronosPipeline.from_pretrained('amazon/chronos-t5-base', device_map=DEVICE, dtype=torch.float32)
    metrics = evaluate_model(
        pipeline,
        str(eval_tsf),
        PREDICTION_LENGTH,
        n_series=N_SERIES,
        num_samples=20,
        freq_str=freq,
        seed=SEED,
        save_dir=run_dir,
    )
    (run_dir / 'results_summary.json').write_text(json.dumps([{'Model': ZERO_SHOT_LABEL, **metrics}], indent=2))

def eval_chronos_checkpoint(run_dir: Path, checkpoint: str, label: str) -> None:
    from src.chronos_finetune import evaluate_model, load_finetuned_pipeline

    run_dir.mkdir(parents=True, exist_ok=True)
    ckpt = require_path(checkpoint, 'adapter_model.safetensors')
    eval_tsf, freq = make_chronos_eval_tsf(run_dir)
    pipeline = load_finetuned_pipeline(str(ckpt))
    metrics = evaluate_model(
        pipeline,
        str(eval_tsf),
        PREDICTION_LENGTH,
        n_series=N_SERIES,
        num_samples=20,
        freq_str=freq,
        seed=SEED,
        save_dir=run_dir,
    )
    (run_dir / 'results_summary.json').write_text(json.dumps([{'Model': label, **metrics}], indent=2))

def baseline_eval_args(run_dir: Path, extra: list[str]) -> list[str]:
    return [
        'scripts/timesfm_baseline.py',
        '--tsf-path', str(TSF_PATH),
        '--output-dir', str(run_dir),
        '--prediction-length', str(PREDICTION_LENGTH),
        '--context-length', str(CONTEXT_LENGTH),
        '--eval-n-series', str(N_SERIES),
        '--seed', str(SEED),
        '--no-plot',
        *extra,
    ]

def ce_eval_args(run_dir: Path, checkpoint: str, n_bins: int) -> list[str]:
    return [
        'scripts/timesfm_ce_finetune.py',
        '--tsf-path', str(TSF_PATH),
        '--output-dir', str(run_dir),
        '--ce-checkpoint', str(require_path(checkpoint, 'ce_head.pt')),
        '--skip-zeroshot',
        '--skip-finetune',
        '--prediction-length', str(PREDICTION_LENGTH),
        '--context-length', str(CONTEXT_LENGTH),
        '--eval-n-series', str(N_SERIES),
        '--n-bins', str(n_bins),
        '--seed', str(SEED),
        '--no-plot',
    ]

EVAL_SPECS = {
    'chronos_zeroshot': {
        'group': 'chronos', 'title': 'Chronos zero-shot', 'out': 'chronos_zeroshot',
        'expected': 'WQL=1.5441, MASE=1.6617', 'runner': eval_chronos_zeroshot,
    },
    'chronos_ce_lora': {
        'group': 'chronos', 'title': 'Chronos CE+LoRA', 'out': 'chronos_ce_lora',
        'expected': 'WQL=1.2244, MASE=1.3806',
        'runner': lambda d: eval_chronos_checkpoint(d, 'output/chronos_finetune_base_lora_repro_20260606_011352/baseline_ce_lora/checkpoint-final', 'CE+LoRA Fine-tune'),
    },
    'chronos_bin_mse_lora': {
        'group': 'chronos', 'title': 'Chronos bin-MSE+LoRA', 'out': 'chronos_bin_mse_lora',
        'expected': 'WQL=1.2651, MASE=1.4176',
        'runner': lambda d: eval_chronos_checkpoint(d, 'output/chronos_finetune_base_lora_repro_20260606_011352/exp1a_bin_mse_lora/checkpoint-final', 'bin-MSE+LoRA Fine-tune'),
    },
    'chronos_mse_ce_lora': {
        'group': 'chronos', 'title': 'Chronos bin-MSE+CE+LoRA lambda=1e-4', 'out': 'chronos_mse_ce_lora',
        'expected': 'WQL=1.2638, MASE=1.4147',
        'runner': lambda d: eval_chronos_checkpoint(d, 'output/chronos_exp2_mse_ce_lora_lambda1e4_repro_20260606_013303/exp2_bin_mse_ce_lora/checkpoint-final', 'bin-MSE+CE+LoRA Fine-tune'),
    },
    'chronos_w1_lora': {
        'group': 'chronos', 'title': 'Chronos bin-W1+LoRA', 'out': 'chronos_w1_lora',
        'expected': 'WQL=1.2750, MASE=1.3151',
        'runner': lambda d: eval_chronos_checkpoint(d, 'output/chronos_remaining_losses_repro_20260606_014915/exp4_bin_wass1_lora/checkpoint-final', 'bin-W1+LoRA Fine-tune'),
    },
    'chronos_w2_lora': {
        'group': 'chronos', 'title': 'Chronos bin-W2+LoRA', 'out': 'chronos_w2_lora',
        'expected': 'WQL=1.3252, MASE=1.3352',
        'runner': lambda d: eval_chronos_checkpoint(d, 'output/chronos_remaining_losses_repro_20260606_014915/exp5_bin_wass2_lora/checkpoint-final', 'bin-W2+LoRA Fine-tune'),
    },
    'chronos_crps_lora': {
        'group': 'chronos', 'title': 'Chronos bin-CRPS+LoRA', 'out': 'chronos_crps_lora',
        'expected': 'WQL=1.2325, MASE=1.3713',
        'runner': lambda d: eval_chronos_checkpoint(d, 'output/chronos_remaining_losses_repro_20260606_014915/exp7_bin_crps_lora/checkpoint-final', 'bin-CRPS+LoRA Fine-tune'),
    },
    'chronos_ordinal_lora': {
        'group': 'chronos', 'title': 'Chronos bin-OrdinalCE+LoRA', 'out': 'chronos_ordinal_lora',
        'expected': 'WQL=1.2357, MASE=1.3765',
        'runner': lambda d: eval_chronos_checkpoint(d, 'output/chronos_remaining_losses_repro_20260606_014915/exp8_bin_ordinal_ce_lora/checkpoint-final', 'bin-OrdinalCE+LoRA Fine-tune'),
    },
    'chronos_huber16_lora': {
        'group': 'chronos', 'title': 'Chronos bin-Huber(16)+LoRA', 'out': 'chronos_huber16_lora',
        'expected': 'WQL=1.2562, MASE=1.3803',
        'runner': lambda d: eval_chronos_checkpoint(d, 'output/chronos_remaining_losses_repro_20260606_014915/exp_6a_bin_huber_16_lora/checkpoint-final', 'bin-Huber(16)+LoRA Fine-tune'),
    },
    'timesfm_base_zeroshot': {
        'group': 'timesfm', 'title': 'TimesFM base zero-shot', 'out': 'timesfm_base_zeroshot',
        'expected': 'WQL=1.1899, MASE=1.1688',
        'args': lambda d: baseline_eval_args(d, ['--skip-finetune']),
    },
    'timesfm_base_lora': {
        'group': 'timesfm', 'title': 'TimesFM base + LoRA', 'out': 'timesfm_base_lora',
        'expected': 'WQL=1.2058, MASE=1.1410',
        'args': lambda d: baseline_eval_args(d, ['--adapter-path', str(require_path('output/timesfm_finetune_repro_20260606_023805', 'adapter_model.safetensors')), '--skip-zeroshot', '--skip-finetune']),
    },
    'timesfm_original_head_eval': {
        'group': 'timesfm', 'title': 'TimesFM original head direct', 'out': 'timesfm_original_head_eval',
        'expected': 'WQL=1.2795, MASE=1.2389',
        'args': lambda d: baseline_eval_args(d, ['--mode', 'head', '--forecast-head-checkpoint', str(require_path('output/timesfm_original_head_legacy64_repro_20260606_024642', 'forecast_head.pt')), '--skip-zeroshot', '--skip-finetune']),
    },
    'timesfm_original_head_lora': {
        'group': 'timesfm', 'title': 'TimesFM original head + LoRA', 'out': 'timesfm_original_head_lora',
        'expected': 'WQL=1.2049, MASE=1.1468',
        'args': lambda d: baseline_eval_args(d, ['--adapter-path', str(require_path('output/timesfm_lora_from_original_head_legacy64_repro_20260606_030307', 'adapter_model.safetensors')), '--skip-zeroshot', '--skip-finetune']),
    },
    'timesfm_ce64_eval': {
        'group': 'timesfm', 'title': 'TimesFM CE64 direct', 'out': 'timesfm_ce64_eval',
        'expected': 'WQL=1.4605, MASE=1.3809',
        'args': lambda d: ce_eval_args(d, 'output/timesfm_ce_pretrain_strict64_repro_20260606_030942', 64),
    },
    'timesfm_ce64_lora_low': {
        'group': 'timesfm', 'title': 'TimesFM CE64 + LoRA low LR', 'out': 'timesfm_ce64_lora_low',
        'expected': 'WQL=1.2308, MASE=1.1162',
        'args': lambda d: ce_eval_args(d, 'output/timesfm_ce_legacy64_lora_repeat_repro_20260606_131025', 64),
    },
    'timesfm_ce64_lora_high': {
        'group': 'timesfm', 'title': 'TimesFM CE64 + LoRA old high LR', 'out': 'timesfm_ce64_lora_high',
        'expected': 'WQL=1.3459, MASE=1.2697',
        'args': lambda d: ce_eval_args(d, 'output/timesfm_ce_legacy64_lora_repro_20260606_032337', 64),
    },
    'timesfm_ce256_eval': {
        'group': 'timesfm', 'title': 'TimesFM CE256 direct', 'out': 'timesfm_ce256_eval',
        'expected': 'WQL=1.3624, MASE=1.2862',
        'args': lambda d: ce_eval_args(d, 'output/timesfm_ce_pretrain_256_repro_20260606_033145', 256),
    },
    'timesfm_ce256_lora_1e5': {
        'group': 'timesfm', 'title': 'TimesFM CE256 + LoRA head_lr=1e-5', 'out': 'timesfm_ce256_lora_1e5',
        'expected': 'WQL=1.2691, MASE=1.2336',
        'args': lambda d: ce_eval_args(d, 'output/timesfm_ce_256_lora_head1e5_repro_20260606_125004', 256),
    },
    'timesfm_ce256_lora_5e5': {
        'group': 'timesfm', 'title': 'TimesFM CE256 + LoRA head_lr=5e-5', 'out': 'timesfm_ce256_lora_5e5',
        'expected': 'WQL=1.3094, MASE=1.2605',
        'args': lambda d: ce_eval_args(d, 'output/timesfm_ce_256_lora_repro_20260606_123909', 256),
    },
    'timesfm_new64_eval': {
        'group': 'timesfm', 'title': 'TimesFM new64 large direct', 'out': 'timesfm_new64_eval',
        'expected': 'WQL=4.5760, MASE=5.5845',
        'args': lambda d: ce_eval_args(d, 'output/timesfm_ce_staged_pretrain_64bin_repro_20260606_041027/01_large', 64),
    },
    'timesfm_new64_lora': {
        'group': 'timesfm', 'title': 'TimesFM new64 large + LoRA', 'out': 'timesfm_new64_lora',
        'expected': 'WQL=1.4641, MASE=1.3968',
        'args': lambda d: ce_eval_args(d, 'output/timesfm_ce_new64_large_lora_repro_20260606_124318', 64),
    },
}

def list_experiments(group: str | None = None) -> pd.DataFrame:
    rows = []
    for name, spec in EVAL_SPECS.items():
        if group is None or spec['group'] == group:
            rows.append({'name': name, 'group': spec['group'], 'title': spec['title'], 'expected': spec['expected']})
    df = pd.DataFrame(rows)
    display(df)
    return df

def run_eval(name: str) -> Path:
    if name not in EVAL_SPECS:
        raise KeyError(f'Unknown eval name: {name}')
    spec = EVAL_SPECS[name]
    run_dir = EVAL_ROOT / spec['out']
    print('\n' + '=' * 72)
    print(spec['title'])
    print(f'Expected: {spec["expected"]}')
    print(f'Output: {rel(run_dir)}')
    if 'runner' in spec:
        spec['runner'](run_dir)
    else:
        run_process(spec['args'](run_dir), run_dir)
    show_results(run_dir)
    plot_predictions(run_dir, spec['title'])
    return run_dir

def run_eval_group(group: str) -> None:
    for name, spec in EVAL_SPECS.items():
        if group == 'all' or spec['group'] == group:
            run_eval(name)

def show_eval_summary() -> pd.DataFrame:
    rows = []
    for summary in sorted(EVAL_ROOT.rglob('results_summary.json')):
        for row in json.loads(summary.read_text()):
            rows.append({'run': rel(summary.parent), **row})
    df = pd.DataFrame(rows)
    display(df)
    return df

print(f'PROJECT_ROOT={rel(PROJECT_ROOT)}')
print(f'EVAL_ROOT={rel(EVAL_ROOT)}')
list_experiments()


## Chronos Eval

只加载 `amazon/chronos-t5-base` 或已有 Chronos LoRA checkpoint 做评估。

In [ ]:
run_eval_group('chronos')


## TimesFM Eval

只加载 `google/timesfm-2.5-200m-pytorch` 和已有 head / LoRA / CE checkpoint 做评估。

In [ ]:
run_eval_group('timesfm')


## Single Eval

如果只想看一个实验，把下面注释取消并替换实验名。

In [ ]:
# run_eval('timesfm_ce64_lora_low')
list_experiments()


## Summary

汇总本次 eval notebook 运行产生的结果。

In [ ]:
show_eval_summary()
